> 在线讲义：[第 26 章 · 读源码：nanoGPT 和 nanochat 逐项对比](https://wikieden.github.io/beyond-transformer/lessons/26-nanochat.html) · 本书《超越 Transformer》

# 第 26 章 · 读源码：nanoGPT 和 nanochat 逐项对比

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/26-nanochat.html) · 仓库里 `beyond-transformer/notes/lessons/26-nanochat.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 激活 | GELU vs ReLU² | `gelu`、`relu2` | ReLU²(2) = 4 |
| 第二步 norm、soft-cap | 无参数 RMS；15·tanh | `rms_norm_noparam`、`softcap` | 100 → 15.0 |
| 第二步 窗口 | SSSL | `window_sizes()` | [512, 512, 512, 2048] × 3 |
| 第三步 小技巧 | 初始化 | `resid_x0_init()`、`has_ve(i)` | 1.15 → 1.05；第 1、3 … 11 层 |

ReLU² 激活、无参数 RMSNorm、输出 logit 的 soft-cap = 15、每层的窗口模式 SSSL、残差 / x0 系数的初始化、哪些层有 value embedding。
差异表写进自己的笔记、在 a1 模型上实现这些改动是里程碑 T-M5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `gelu`**：小工具。

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))

**函数 `relu2`**：nanochat 的 MLP 激活：F.relu(x).square()。

In [ ]:
def relu2(x):
    """nanochat 的 MLP 激活：F.relu(x).square()。"""
    return np.maximum(x, 0) ** 2

**函数 `rms_norm_noparam`**：F.rms_norm(x, (d,))：没有可学的 g，也没有 bias。

In [ ]:
def rms_norm_noparam(x, eps=1e-6):
    """F.rms_norm(x, (d,))：没有可学的 g，也没有 bias。"""
    return x / np.sqrt((x * x).mean() + eps)

**函数 `softcap`**：小工具。

In [ ]:
def softcap(x, cap=15.0):
    return cap * np.tanh(np.asarray(x, float) / cap)

**函数 `window_sizes`**：S = 四分之一上下文（向上取到 128 的倍数），L = 全上下文；按 pattern 平铺，最后一层强制 L。

In [ ]:
def window_sizes(n_layer=12, seq=2048, pattern="SSSL"):
    """S = 四分之一上下文（向上取到 128 的倍数），L = 全上下文；按 pattern 平铺，最后一层强制 L。"""
    short = -(-seq // 4 // 128) * 128
    ws = [seq if pattern[i % len(pattern)] == "L" else short for i in range(n_layer)]
    ws[-1] = seq
    return ws

**函数 `resid_x0_init`**：小工具。

In [ ]:
def resid_x0_init(n_layer=12):
    resid = [1.15 - 0.10 * i / max(n_layer - 1, 1) for i in range(n_layer)]
    x0 = [0.20 - 0.15 * i / max(n_layer - 1, 1) for i in range(n_layer)]
    return resid, x0

**函数 `has_ve`**：小工具。

In [ ]:
def has_ve(i, n_layer=12):
    return i % 2 == (n_layer - 1) % 2

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 MLP 激活：GELU（nanoGPT）vs ReLU²（nanochat）")
for x in (-2.0, -0.5, 0.5, 1.0, 2.0):
    print(f"  x = {x:+.1f}：GELU {gelu(x):+.4f}，ReLU² {relu2(x):.4f}")
print("\n第二步 norm 不带参数")
v = np.array([1.0, 2.0, 3.0, 6.0])
print(f"  {v.tolist()} → {np.round(rms_norm_noparam(v), 4).tolist()}（nanoGPT 的 LayerNorm 有 γ、β）")

In [ ]:
print("\n第三步 输出 logit soft-cap = 15")
print(f"  {[5, 15, 30, 100]} → {np.round(softcap([5, 15, 30, 100]), 2).tolist()}")

In [ ]:
print("\n第四步 每层的窗口（12 层、上下文 2048、模式 SSSL）")
print(f"  {window_sizes()}")

In [ ]:
print("\n第五步 残差流上的两个可学标量（初始化）")
r, x0 = resid_x0_init()
print(f"  resid_lambda：第 1 层 {r[0]:.2f} → 第 12 层 {r[-1]:.2f}；x0_lambda：{x0[0]:.2f} → {x0[-1]:.2f}")
print(f"  有 value embedding 的层：{[i for i in range(12) if has_ve(i)]}")

## 3. 代码里才看得到的

- **注释和代码对不上**：nanochat 的窗口计算旁边注释写着“2048 → 768”，可按代码 `-(-2048 // 4 // 128) * 128` 算出来是 512。读源码时以代码为准，注释可能是旧版本留下的。
- **RoPE 的写法是“前后两半配对”**：nanochat 的 `apply_rotary_emb`（L57–L65）把向量切成前后两半，和 a1 的相邻配对不同（第 15 章），而且注释说转的是 −θ；只要训练和推理一致就没问题。

## 4. 自己改一改

- 把 `window_sizes` 的模式改成 "SL"，12 层里有几层全局？
- 把 `resid_x0_init` 的层数改成 24，第 12 层的两个系数各是多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 同一本菜谱重写：位置换 RoPE，norm 换无参 RMS、嵌入后也 norm，激活换 ReLU 平方，注意力加 GQA、QK-norm、三短一长，输出 cap 十五。
> 初始化残差投影为零，优化器矩阵用 Muon。
> 速通小技巧：残差系数、x0 混入、值嵌入、smear、backout，分开看。
> 读源码：先特性、再 forward、最后初始化和优化器。